# AyosBayan - Pothole Detection Model Training
## Google Colab Notebook for YOLOv8 Training

This notebook trains the YOLOv8 model for pothole detection using Google Colab's free GPU.

---

## Overview

This notebook follows the complete ML workflow from the README_v2.md:

1. **Image Collection** - Setting up the dataset
2. **Data Processing** - Preprocessing and augmentation
3. **Feature Engineering** - Automatic (YOLOv8 does this)
4. **Model Selection** - YOLOv8 (nano variant)
5. **Training** - Transfer learning with pre-trained weights
6. **Evaluation** - Checking model performance
7. **Model Versioning** - Saving the trained model

---

## What is YOLOv8?

**YOLO (You Only Look Once)** is a state-of-the-art object detection algorithm.

**YOLOv8** is the latest version from Ultralytics, featuring:
- ⚡ Fast inference (< 2 seconds on CPU)
- 🎯 High accuracy for road damage detection
- 📦 Easy to use Python API
- 🔄 Transfer learning from COCO dataset

---

## Why Google Colab?

Google Colab provides:
- **Free GPU** (NVIDIA T4 or similar)
- **Pre-installed libraries** (Ultralytics, PyTorch)
- **Easy sharing** with team members
- **No local setup** required

---

## Author
**Federex** - AyosBayan Research Project

Version: 1.0 | Last Updated: April 2026

---

## Step 1: Setup and Installation

First, let's install the required libraries and check our environment.

In [ ]:
# Step 1a: Check GPU availability
# This helps us verify we're running on a GPU-enabled machine
# GPU makes training much faster (10-100x vs CPU)

# Import the GPU checking function
import torch

# Check if CUDA is available (NVIDIA GPU)
print("=" * 60)
print("GPU CHECK")
print("=" * 60)
print(f"CUDA available: {torch.cuda.is_available()}")

# If GPU is available, show details
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Warning: No GPU detected. Training will be slow on CPU.")
    print("Tip: Go to Runtime → Change runtime type → GPU")

print("=" * 60)

In [ ]:
# Step 1b: Install required libraries
# We need Ultralytics for YOLOv8
# The ! prefix runs commands in the shell (bash)

print("Installing required libraries...")
!pip install ultralytics==8.3.0

# Verify installation
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

---

## Step 2: Dataset Setup

Now let's set up the pothole dataset. You have two options:

### Option A: Download from Roboflow (Recommended)

1. Go to [Roboflow](https://roboflow.com/) and create an account
2. Search for "pothole detection" dataset
3. Export in **YOLO format**
4. Copy the download command (curl -L)

### Option B: Use Sample Data

For testing purposes, we'll use a sample dataset.

---

**Note:** For this demonstration, we'll use the Roboflow pothole dataset.

In [ ]:
# Step 2a: Create dataset directory structure
# YOLOv8 expects a specific folder structure:
# dataset/
# ├── train/images/
# ├── train/labels/
# ├── val/images/
# ├── val/labels/
# └── data.yaml

import os

# Create the base dataset directory
dataset_dir = '/content/pothole_dataset'
os.makedirs(dataset_dir, exist_ok=True)

# Create train/val subdirectories
for split in ['train', 'val']:
    os.makedirs(f'{dataset_dir}/{split}/images', exist_ok=True)
    os.makedirs(f'{dataset_dir}/{split}/labels', exist_ok=True)

print("Dataset directory structure created!")
print(f"Directory: {dataset_dir}")

In [ ]:
# Step 2b: Create data.yaml configuration file
# This tells YOLO where to find images and what classes to detect

data_yaml_content = """
# Pothole Dataset Configuration for YOLOv8

# Dataset paths (absolute paths for Colab)
train: /content/pothole_dataset/train/images
val: /content/pothole_dataset/val/images
test: /content/pothole_dataset/test/images

# Number of classes
nc: 1

# Class names
names:
  0: pothole
"""

# Write to file
data_yaml_path = '/content/pothole_dataset/data.yaml'
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml_content)

print(f"data.yaml created at: {data_yaml_path}")
print("\nContent:")
print(data_yaml_content)

In [ ]:
# Step 2c: Download pothole dataset from Roboflow
# 
# REPLACE the API key below with your own!
# 
# To get your API key:
# 1. Go to Roboflow -> Your Name -> Settings
# 2. Copy your Private API Key
# 
# For this demo, we'll use a public dataset

# NOTE: Replace with YOUR Roboflow API key
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # <-- REPLACE THIS!

# If you don't have an API key, you can use this alternative:
# Download from a public Kaggle dataset or GitHub repository

print("=" * 60)
print("DATASET DOWNLOAD")
print("=" * 60)
print("\nOption 1: Roboflow (Recommended)")
print("  1. Get your API key from https://roboflow.com/settings/api")
print("  2. Find a pothole dataset: https://universe.roboflow.com/search?q=pothole")
print("  3. Export in YOLO format and copy the curl command")
print("\nOption 2: Use sample command below")
print("  (This is an example - find a real dataset first)")
print("=" * 60)

# Example download command (replace with actual)
# !curl -L "https://universe.roboflow.com/ds/YOUR_DATASET_KEY" > dataset.zip
# !unzip dataset.zip -d /content/pothole_dataset/
# !rm dataset.zip

In [ ]:
# Step 2d: Test your Roboflow API key
# This verifies that your API key is valid and working

print("=" * 60)
print("TESTING ROBOFLOW API KEY")
print("=" * 60)

# Install roboflow library
!pip install roboflow -q

from roboflow import Roboflow

try:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    
    # Try to connect - this will fail if key is invalid
    workspace = rf.workspace()
    projects = workspace.projects()
    
    print("✓ SUCCESS: Your Roboflow API key is valid!")
    print(f"\\n  You have {len(projects)} project(s)")
    print("\\n  Next step: Find or upload a pothole dataset!")
    
except Exception as e:
    print("✗ FAILED: Invalid or expired Roboflow API key")
    print(f"\\n  Error: {str(e)}")
    print("\\n  To fix:")
    print("  1. Go to https://app.roboflow.com/settings/api")
    print("  2. Copy your API key")
    print("  3. Replace ROBOFLOW_API_KEY in the cell above")

print("=" * 60)


In [ ]:
# Step 2e: Download pothole dataset
# Use the Roboflow API to download your dataset

print("=" * 60)
print("DOWNLOADING POTHOLE DATASET")
print("=" * 60)

# Method 1: Download from Roboflow (if you have a project)
# Replace with your workspace/project info

try:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    
    # Option A: If you created a project in Roboflow
    # workspace = rf.workspace()
    # project = workspace.project("your-project-name")
    # version = project.version(1)
    # dataset = version.download("yolov8", location="/content/pothole_dataset")
    
    # Option B: Download a public dataset
    # Find public datasets at: https://universe.roboflow.com/
    # Example: pothole detection datasets
    
    print("\nTo download a dataset:")
    print("  1. Go to https://universe.roboflow.com/")
    print("  2. Search for 'pothole'")
    print("  3. Click on a dataset -> Deploy -> Download Dataset")
    print("  4. Copy the curl command and paste below")
    print("\nOr manually upload images to /content/pothole_dataset/train/images/")
    
    print("\nSkipping download (no dataset specified)")
    
except Exception as e:
    print(f"Error: {str(e)}")

# Method 2: Alternative - Use curl with a public Roboflow dataset
# Uncomment and run one of these commands:

# Download sample pothole dataset (public)
# !curl -L "https://universe.roboflow.com/ds/YOUR_DATASET_ID" > dataset.zip
# !unzip dataset.zip -d /content/pothole_dataset/
# !rm dataset.zip

print("=" * 60)


In [ ]:
    print("\n  Workspace loaded successfully!")


---

## Step 3: Load Pre-trained YOLOv8 Model

We use **transfer learning** by starting with a model pre-trained on the COCO dataset.
This model already knows basic visual features, so we only need to fine-tune for potholes.

### YOLOv8 Variants

| Model | Size | Speed | Accuracy | Use Case |
|-------|------|-------|----------|----------|
|**yolov8n**|3.9MB|⚡⚡⚡⚡⚡|⬜⬜|Fastest - mobile/deployment|
|**yolov8s**|7.7MB|⚡⚡⚡⚡|⬜⬜⬜|Balanced (we use this) |
|**yolov8m**|25.9MB|⚡⚡⚡|⬜⬜⬜⬜|Higher accuracy |
|**yolov8l**|51.0MB|⚡⚡|⬜⬜⬜⬜⬜|Highest accuracy |

We recommend **yolov8n** (nano) or **yolov8s** (small) for this project.

In [ ]:
# Step 3: Load pre-trained YOLOv8 model
# 
# The .pt file contains pre-trained weights from COCO dataset
# This is transfer learning - we start with a model that already knows
# how to detect many objects (cars, roads, etc.)

from ultralytics import YOLO

# Choose model variant
# yolov8n = nano (fastest, smallest)
# yolov8s = small (balanced)
# yolov8m = medium (more accurate)
MODEL_VARIANT = "yolov8n"  # Change to "yolov8s" for more accuracy

print("=" * 60)
print(f"Loading pre-trained {MODEL_VARIANT}.pt")
print("=" * 60)

# Load the model
# This automatically downloads the pre-trained weights if not present
model = YOLO(f'{MODEL_VARIANT}.pt')

print(f"✓ {MODEL_VARIANT}.pt loaded successfully!")
print(f"\nModel info:")
print(f"  - Type: {MODEL_VARIANT}")
print(f"  - Pre-trained on: COCO dataset (80+ classes)")
print(f"  - Ready for fine-tuning: Yes")

---

## Step 4: Configure Training Parameters

Now let's configure the training parameters. These control how the model learns.

### Key Parameters Explained

### **Epochs**: 50-100
One epoch = one complete pass through all training images.
50 epochs is usually enough for fine-tuning with pre-trained weights.

### **Image Size**: 640
YOLOv8 works best with 640×640 pixel images.
Smaller sizes are faster but less accurate.

### **Batch Size**: 16
Number of images processed at once.
16 is good for most GPUs. Reduce if you get memory errors.

### **Learning Rate**: Auto
YOLOv8 automatically sets a good learning rate.
No need to change this.

### **Augmentation**: Enabled
YOLOv8 automatically augments data (rotation, flip, brightness, etc.)
This helps the model generalize to different conditions.

In [ ]:
# Step 4: Configure training parameters
# These are the hyperparameters for training

# Training configuration
EPOCHS = 50              # Number of training epochs
IMAGE_SIZE = 640        # Input image size (YOLOv8 standard)
BATCH_SIZE = 16         # Images per batch (adjust for GPU memory)
PATIENCE = 20           # Early stopping patience

# Save configuration
MODEL_NAME = "ayosbayan_pothole"

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)
print(f"  Epochs: {EPOCHS}")
print(f"  Image Size: {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Early Stopping Patience: {PATIENCE}")
print(f"  Model Name: {MODEL_NAME}")
print("=" * 60)

---

## Step 5: Start Training

Now we start the actual training! This will:

1. Load the pre-trained model
2. Fine-tune on our pothole dataset
3. Validate after each epoch
4. Save the best model

**Training time:** ~10-30 minutes on GPU depending on dataset size and epochs.

---

### What happens during training?

The model will:
- See all training images each epoch
- Learn to predict bounding boxes around potholes
- Be evaluated on validation data after each epoch
- Save the "best" model (highest validation accuracy)

### Monitoring

Watch for these metrics:
- **box_loss**: Bounding box prediction error (lower is better)
- **cls_loss**: Classification error (lower is better)
- **precision**: Accuracy of positive predictions
- **recall**: Ability to find all potholes
- **mAP@50**: Mean Average Precision at 50% IOU

In [ ]:
# Step 5: Start training
# This is where the actual learning happens

# Disable wandb logging (wandb doesn't allow / in project names)
import os
os.environ["WANDB_DISABLED"] = "true"

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print("\nThis will take 10-30 minutes depending on GPU and epochs.")
print("\nTraining progress:")
print("-" * 60)

# Run training
# The .train() method handles:
# - Data loading
# - Forward/backward passes
# - Validation
# - Checkpoint saving

results = model.train(
    # Data configuration
    data='/content/pothole_dataset/data.yaml',
    
    # Training parameters
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    
    # Save settings
    project='/content/training_results',
    name=MODEL_NAME,
    exist_ok=True,
    
    # Device
    device=0,  # Use GPU 0 (set to 'cpu' for CPU training)
    
    # Other settings
    workers=8,
    amp=True,  # Automatic Mixed Precision (faster training)
    verbose=True,
)

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)


---

## Step 6: Evaluate Model Performance

After training, let's evaluate the model's performance.

### Key Metrics Explained

### **Precision** (> 0.85)
Percentage of correct positive predictions.
High precision = fewer false positives (fewer fake reports flagged wrongly)

### **Recall** (> 0.85)
Ability to find all actual positives.
High recall = we catch most real potholes

### **mAP@50** (> 0.90)
Mean Average Precision at 50% IOU.
Overall detection quality - higher is better

### **F1-Score** (> 0.85)
Balance between precision and recall.
Harmonic mean of both metrics

In [ ]:
# Step 6a: Display training results
# Show the final metrics from training

print("=" * 60)
print("TRAINING RESULTS")
print("=" * 60)

# Get the results dictionary
print("\nFinal Metrics:")
print(f"  - Precision: {results.results_dict.get('metrics/precision(B)', 0):.4f}")
print(f"  - Recall: {results.results_dict.get('metrics/recall(B)', 0):.4f}")
print(f"  - mAP@50: {results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"  - mAP@50-95: {results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")

print("\n" + "-" * 60)
print("Training Metrics Explanation:")
print("-" * 60)
print("Precision > 0.85: Low false positives (fewer fake reports flagged wrongly)")
print("Recall > 0.85: Catch most real potholes")
print("mAP@50 > 0.90: Overall detection quality")
print("F1-Score > 0.85: Balance of precision & recall")

In [ ]:
# Step 6b: Find and verify the trained model
# The best model is saved in the training results folder

import os

# Path to the best model
best_model_path = f'/content/training_results/{MODEL_NAME}/weights/best.pt'

print("=" * 60)
print("TRAINED MODEL")
print("=" * 60)

# Check if the model exists
if os.path.exists(best_model_path):
    # Get file size
    file_size = os.path.getsize(best_model_path) / (1024 * 1024)  # MB
    print(f"✓ Best model saved!")
    print(f"  Path: {best_model_path}")
    print(f"  Size: {file_size:.2f} MB")
else:
    print(f"Model not found at: {best_model_path}")
    # Try the last epoch model
    last_model_path = f'/content/training_results/{MODEL_NAME}/weights/last.pt'
    if os.path.exists(last_model_path):
        print(f"Using last epoch model: {last_model_path}")
        best_model_path = last_model_path

---

## Step 7: Download the Trained Model

Now let's download the trained model to use in our local inference and API.

### Next Steps

1. **Download** the best.pt file
2. **Copy** it to your local project
3. **Test** with local inference: `python src/inference.py --image test.jpg`
4. **Run** the API: `uvicorn src.api:app --port 8000`

In [ ]:
# Step 7: Download the trained model
# This allows you to use the model locally

from google.colab import files

print("=" * 60)
print("DOWNLOAD MODEL")
print("=" * 60)

print("\nClick the download button below to download best.pt")
print("Or run this code to download:")

# Download the model
files.download(best_model_path)

print(f"\nDownloaded: {best_model_path}")

In [ ]:
# Alternative: Copy to Google Drive (optional)
# If you have Google Drive mounted, you can save there

# from google.colab import drive
# drive.mount('/content/drive')
# 
# Copy model to Drive
# import shutil
# shutil.copy(best_model_path, '/content/drive/MyDrive/ayosbayan/best.pt')

---

## Step 8: Test Inference (Optional)

Let's test the trained model with a sample image to verify it works.

### How Inference Works

1. **Load** the trained model
2. **Predict** on a new image
3. **Get** bounding boxes and confidence scores
4. **Process** results

### Expected Output

- `detected`: true/false (pothole found?)
- `confidence`: 0.0-1.0 (how confident)
- `bounding_box`: coordinates (if detected)
- `is_potential_fake`: true/false (needs review?)

In [ ]:
# Step 8: Test inference with a sample image
# This demonstrates how to use the trained model

# Load the trained model
trained_model = YOLO(best_model_path)

# Test on a sample image from validation set
# (Replace with actual test image)

print("=" * 60)
print("INFERENCE TEST")
print("=" * 60)

# Run inference on validation image
# You can also test on your own images

# Example: Use first validation image
val_images_dir = '/content/pothole_dataset/val/images'
if os.path.exists(val_images_dir) and os.listdir(val_images_dir):
    test_image = os.listdir(val_images_dir)[0]
    test_image_path = f'{val_images_dir}/{test_image}'
    
    print(f"\nTesting on: {test_image}")
    
    # Run inference
    results = trained_model.predict(
        source=test_image_path,
        conf=0.5,  # Confidence threshold
        save=True  # Save annotated image
    )
    
    # Process results
    result = results[0]
    
    # Display results
    print(f"\nDetection Results:")
    print(f"  - Detected: {len(result.boxes) > 0}")
    
    if len(result.boxes) > 0:
        box = result.boxes[0]
        confidence = float(box.conf[0])
        class_id = int(box.cls[0])
        class_name = result.names[class_id]
        
        print(f"  - Class: {class_name}")
        print(f"  - Confidence: {confidence:.2%}")
        
        # Bounding box
        xyxy = box.xyxy[0].cpu().numpy()
        print(f"  - Bounding Box: [{xyxy[0]:.0f}, {xyxy[1]:.0f}, {xyxy[2]:.0f}, {xyxy[3]:.0f}]")
    
    # Show the saved image
    from IPython.display import Image, display
    print("\nAnnotated image saved to:")
    print(f"  {results[0].save_dir}")
else:
    print("No validation images available for testing.")
    print("Upload test images to test inference.")

---

## Summary

Congratulations! You've trained a YOLOv8 pothole detection model.

### What We Accomplished

1. ✅ Set up Google Colab environment
2. ✅ Loaded pre-trained YOLOv8 model
3. ✅ Configured training parameters
4. ✅ Trained on pothole dataset
5. ✅ Evaluated model performance
6. ✅ Downloaded trained model

### Next Steps for Local Deployment

1. **Download** best.pt from Colab
2. **Copy** to your local project root
3. **Test** inference: `python src/inference.py --image test.jpg`
4. **Run** API: `uvicorn src.api:app --port 8000`
5. **Integrate** with your web application

---

### Model Performance Targets

| Metric | Target | Your Model |
|--------|--------|------------|
Precision | > 0.85 | Check training results |
Recall | > 0.85 | Check training results |
mAP@50 | > 0.90 | Check training results |
F1-Score | > 0.85 | Check training results |
Inference Speed | < 2 sec | ~0.1 sec on GPU |

---

### Troubleshooting

**Low Accuracy?**
- Add more training images
- Increase epochs to 100
- Use yolov8s instead of yolov8n
- Add more Philippine-specific images

**Slow Training?**
- Use GPU (Runtime → Change runtime type → GPU)
- Reduce batch size if getting OOM errors
- Use yolov8n (fastest)

**Out of Memory?**
- Reduce batch size to 8 or 4
- Use yolov8n (smaller model)
- Reduce image size to 512 or 320

---

## Author
**Federex** - AyosBayan Research Project

Version: 1.0 | Last Updated: April 2026